# Week 2 — PySpark Fundamentals

**Objetivo:** entender los mecanismos básicos de PySpark: `SparkSession`, DataFrames, lazy evaluation, transformaciones, joins, window functions y particiones.

**Cómo está organizado este notebook:**
1. Verificación del entorno (Spark dentro del contenedor Docker).
2. SparkSession + primer DataFrame.
3. Leer un CSV.
4. Transformaciones básicas (`select`, `filter`, `withColumn`).
5. **Lazy evaluation** — el concepto mental más importante de Spark.
6. `groupBy` y agregaciones.
7. Joins y **revisita del fan-out** que vimos en Week 1.
8. Window Functions.
9. Particiones y `explain()`.
10. Ejercicios.

**Conexión mental con pandas (que ya dominas):** la API de PySpark es _muy_ parecida (`df.filter`, `df.groupBy`, `df.join`). Las **tres diferencias** que tienes que interiorizar son:

| Concepto | pandas | PySpark |
|---|---|---|
| Ejecución | **Eager**: cada línea se ejecuta al instante | **Lazy**: construye un plan, ejecuta sólo al llamar a un *action* |
| Mutabilidad | Puedes mutar in-place (`df['x'] = ...`) | Inmutable. Cada transformación devuelve un DataFrame **nuevo** |
| Distribución | Una máquina, en RAM | Datos en **particiones** sobre N workers. `join`/`groupBy` implican **shuffle** (red) |

Este patrón de "lazy + plan optimizado + ejecución diferida" lo vas a volver a ver en **Airflow** (DAGs de tareas) y **LangChain** (chains diferidas). Aprenderlo aquí te da el modelo mental para los demás.

## Bloque 0 — Verificación del entorno

In [ ]:
# Importamos los módulos básicos de PySpark
import pyspark                                           # paquete raíz, sólo para ver la versión
from pyspark.sql import SparkSession                     # punto de entrada de la API DataFrame
from pyspark.sql import functions as F                   # alias estándar 'F' para funciones (sum, avg, col...)
from pyspark.sql.types import StructType, StructField    # para definir esquemas explícitos (best practice)
from pyspark.sql.types import IntegerType, StringType, DateType, DoubleType

print(f"PySpark version: {pyspark.__version__}")        # debe imprimir 4.1.x

## Bloque 1 — SparkSession y primer DataFrame

**`SparkSession` es el equivalente a `pd` en pandas:** es el punto de entrada que crea DataFrames, lee ficheros, ejecuta SQL. En un proyecto siempre creas **una sola** sesión y la reutilizas.

- `master('local[*]')` → ejecuta Spark localmente usando **todos los cores** disponibles del contenedor.
- En producción este `master` sería la URL de un cluster (YARN/Kubernetes/Standalone), pero el código del DataFrame **no cambia** — esa es la magia de Spark.
- `getOrCreate()` → si ya existe una sesión la reutiliza, si no la crea. Patrón idempotente.

In [ ]:
# Crear (o recuperar) la SparkSession
spark = (
    SparkSession.builder                                 # builder pattern, igual que en Java
    .appName("EUGraphRAG-Week2")                         # nombre visible en la Spark UI (puerto 4040)
    .master("local[*]")                                  # local con todos los cores del contenedor
    .config("spark.sql.shuffle.partitions", "4")         # bajamos las particiones por defecto (200) para datasets pequeños
    .getOrCreate()                                       # crea o recupera
)

# El log level por defecto es WARN; subimos a ERROR para silenciar warnings de demo
spark.sparkContext.setLogLevel("ERROR")

print(f"Spark version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")           # local[*]
print(f"Cores disponibles: {spark.sparkContext.defaultParallelism}")

In [ ]:
# Primer DataFrame: a partir de una lista de tuplas Python
datos = [
    (1, "Regulation", 42, "2024-01-15"),                 # (id, tipo, articulos, fecha)
    (2, "Directive",  18, "2024-03-22"),
    (3, "Decision",    7, "2024-05-09"),
    (4, "Regulation", 65, "2024-07-30"),
    (5, "Directive",  12, "2024-11-04"),
]

# Nombres de columna como lista (Spark infiere los tipos por su cuenta)
columnas = ["doc_id", "tipo", "articulos", "fecha"]

df = spark.createDataFrame(datos, columnas)              # crea el DataFrame distribuido
df.show()                                                # action: ejecuta y muestra el contenido
df.printSchema()                                         # muestra el esquema inferido (todo string excepto los int)

Observa que `fecha` se infirió como `string` (no como `date`). En la práctica esto es un problema: las operaciones de fecha (LAG, diferencias, filtros por rango) no funcionarán sobre strings.

**Best practice:** definir el esquema explícitamente con `StructType`. Lo hacemos al leer el CSV abajo.

## Bloque 2 — Leer un CSV con esquema explícito

En proyectos reales nunca dejas que Spark infiera el esquema (es caro: implica leer el fichero entero antes de procesarlo). Lo defines explícitamente con `StructType`.

In [ ]:
# Generamos un CSV de juguete usando pandas (para no depender de datos externos todavía)
import pandas as pd                                                  # pandas ya viene con la imagen
from pathlib import Path

# OJO con las rutas relativas: NO son relativas al fichero .ipynb, sino al
# *current working directory* (CWD) del kernel de Python. Dentro del contenedor
# Docker el CWD es /home/jovyan; el volumen del host (./data en tu Windows)
# está montado en /home/jovyan/work/data según docker-compose.yml. Usamos ruta
# ABSOLUTA al volumen montado para que el CSV (a) persista al hacer
# `docker compose down` y (b) sea visible desde tu host en ./data/bronze/.
ruta_csv = Path("/home/jovyan/work/data/bronze/sample_legislation.csv")  # volumen montado del host
ruta_csv.parent.mkdir(parents=True, exist_ok=True)                    # crea data/bronze si no existe

pd.DataFrame({
    "doc_id":    [1, 2, 3, 4, 5, 6, 7, 8],
    "tipo":      ["Regulation", "Directive", "Decision", "Regulation",
                  "Directive", "Regulation", "Decision", "Directive"],
    "articulos": [42, 18, 7, 65, 12, 28, 9, 33],
    "fecha":     ["2024-01-15", "2024-03-22", "2024-05-09", "2024-07-30",
                  "2024-11-04", "2025-02-18", "2025-04-11", "2025-08-25"],
}).to_csv(ruta_csv, index=False)                                     # escribe el CSV en disco

print(f"CSV creado en: {ruta_csv.resolve()}")

In [ ]:
# Esquema explícito: una entrada por columna con (nombre, tipo, nullable)
esquema = StructType([
    StructField("doc_id",    IntegerType(), nullable=False),         # entero, obligatorio
    StructField("tipo",      StringType(),  nullable=True),
    StructField("articulos", IntegerType(), nullable=True),
    StructField("fecha",     DateType(),    nullable=True),          # ¡fecha de verdad, no string!
])

df_csv = (
    spark.read                                                       # API de lectura
    .option("header", True)                                          # primera fila = cabecera
    .option("dateFormat", "yyyy-MM-dd")                              # formato para parsear la fecha
    .schema(esquema)                                                 # aplicamos el esquema explícito
    .csv(str(ruta_csv))                                              # ruta del fichero
)

df_csv.printSchema()                                                 # ahora 'fecha' es DateType, no StringType
df_csv.show()

In [ ]:
# describe() → estadísticas básicas, similar a pandas
df_csv.describe(["articulos"]).show()                                # count, mean, stddev, min, max

## Bloque 3 — Transformaciones básicas

Las tres transformaciones que vas a usar el 80% del tiempo:

| Operación | pandas | PySpark |
|---|---|---|
| Seleccionar columnas | `df[['a','b']]` | `df.select('a', 'b')` |
| Filtrar filas | `df[df.x > 10]` | `df.filter(F.col('x') > 10)` |
| Añadir/transformar columna | `df['y'] = df.x * 2` | `df.withColumn('y', F.col('x') * 2)` |

In [ ]:
# select → escoger columnas (devuelve un DataFrame NUEVO; df_csv no cambia)
df_csv.select("doc_id", "tipo").show()

# filter → quedarnos con regulaciones
df_csv.filter(F.col("tipo") == "Regulation").show()

# withColumn → añadir una columna 'articulos_x2'
df_csv.withColumn("articulos_x2", F.col("articulos") * 2).show()

# Encadenadas (estilo idiomático)
(
    df_csv
    .filter(F.col("articulos") > 15)                                 # sólo docs con más de 15 artículos
    .withColumn("categoria", F.when(F.col("articulos") > 30, "grande")
                              .otherwise("medio"))                   # CASE WHEN equivalente
    .select("doc_id", "tipo", "articulos", "categoria")
    .show()
)

## Bloque 4 — Lazy evaluation (el concepto clave)

En pandas, `df[df.x > 10]` ejecuta el filtro al instante.

En PySpark, `df.filter(...)` **no ejecuta nada**. Sólo añade un nodo al **plan lógico** del DataFrame. La ejecución se dispara únicamente cuando llamas a un *action*:

| Tipo | Ejemplos | ¿Dispara ejecución? |
|---|---|---|
| **Transformation** | `select`, `filter`, `withColumn`, `groupBy`, `join` | ❌ No |
| **Action** | `show`, `count`, `collect`, `write`, `toPandas` | ✅ Sí |

**¿Por qué es importante?** Porque Spark **optimiza el plan completo** antes de ejecutarlo (Catalyst optimizer). Por ejemplo, si filtras al final pero el optimizador detecta que puede empujar el filtro hacia abajo (predicate pushdown), lo hace y lee menos datos del disco. **Esto es imposible en pandas porque cada línea ya ha ejecutado.**

In [ ]:
import time                                                          # para medir tiempos

# Encadenamos 4 transformaciones SIN ningún action al final
t0 = time.time()
plan = (
    df_csv
    .filter(F.col("articulos") > 5)
    .withColumn("articulos_doble", F.col("articulos") * 2)
    .filter(F.col("tipo") != "Decision")
    .select("doc_id", "tipo", "articulos_doble")
)
t1 = time.time()
print(f"Encadenar 4 transformaciones tardó: {(t1-t0)*1000:.2f} ms (no se ejecutó nada todavía)")

# Ahora llamamos a un action: AHORA sí se ejecuta el plan entero
t2 = time.time()
plan.show()
t3 = time.time()
print(f"Ejecutar el plan al llamar show() tardó: {(t3-t2)*1000:.2f} ms")

In [ ]:
# Podemos ver el plan optimizado SIN ejecutarlo:
plan.explain(mode="formatted")                                       # imprime el plan físico optimizado

Observa cómo Catalyst ha **combinado los dos filtros** (`articulos > 5 AND tipo != 'Decision'`) en uno solo y los ha empujado al `FileScan`. Eso es exactamente el tipo de optimización que la lazy evaluation permite — y que pandas no puede hacer.

## Bloque 5 — groupBy + agregaciones

La sintaxis es prácticamente idéntica a SQL:

```sql
SELECT tipo, COUNT(*), SUM(articulos), AVG(articulos)
FROM df
GROUP BY tipo;
```

↓

```python
df.groupBy("tipo").agg(F.count("*"), F.sum("articulos"), F.avg("articulos"))
```

In [ ]:
# Agregación simple por tipo de documento
(
    df_csv
    .groupBy("tipo")                                                 # equivalente a GROUP BY tipo
    .agg(
        F.count("*").alias("num_docs"),                              # COUNT(*) AS num_docs
        F.sum("articulos").alias("total_articulos"),                 # SUM(articulos)
        F.avg("articulos").alias("media_articulos"),                 # AVG(articulos)
        F.max("articulos").alias("max_articulos"),                   # MAX(articulos)
    )
    .orderBy(F.desc("total_articulos"))                              # ORDER BY total_articulos DESC
    .show()
)

**⚠️ Aviso sobre performance:** `groupBy` provoca un **shuffle** (redistribuir filas entre particiones para que todas las del mismo grupo acaben en la misma máquina). Es caro. En Spark las optimizaciones más rentables casi siempre pasan por **reducir o evitar shuffles**: pre-agregar, particionar bien, broadcast joins.

## Bloque 6 — Joins y revisita del fan-out

Recordemos Week 1: cuando hicimos `fact_documents JOIN bridge_document_topic`, el `SUM(article_count)` pasó de 2561 a 4106 porque cada documento se replicaba por su número de temas. **El mismo problema aparece igual en Spark.**

Vamos a reproducirlo aquí en pequeño.

In [ ]:
# DataFrame de documentos (5 docs)
df_docs = spark.createDataFrame(
    [
        (1, "Doc A", 10),                                            # (doc_id, titulo, articulos)
        (2, "Doc B", 20),
        (3, "Doc C", 30),
        (4, "Doc D", 40),
        (5, "Doc E", 50),
    ],
    ["doc_id", "titulo", "articulos"],
)

# DataFrame puente: doc 1→1 tema, docs 2 y 3→2 temas cada uno, doc 4→3 temas, doc 5→1 tema
df_bridge = spark.createDataFrame(
    [
        (1, "DIG"),
        (2, "DIG"), (2, "ENV"),
        (3, "FIN"), (3, "ENV"),
        (4, "DIG"), (4, "FIN"), (4, "TRA"),
        (5, "ENV"),
    ],
    ["doc_id", "topic"],
)

print("== Docs (sin JOIN) ==")
df_docs.agg(F.count("*").alias("docs"), F.sum("articulos").alias("total_articulos")).show()
# Esperamos: docs=5, total_articulos=150

In [ ]:
# === EL FAN-OUT EN ACCIÓN ===
df_join = df_docs.join(df_bridge, on="doc_id", how="inner")           # join por doc_id
df_join.show()                                                        # observa cómo doc 2 aparece 2 veces, doc 4 tres veces...

print("== Con JOIN al bridge (¡inflado!) ==")
df_join.agg(F.count("*").alias("docs"), F.sum("articulos").alias("total_articulos")).show()
# Esperamos: docs=9 (filas del bridge), total_articulos=280 (= 10 + 20*2 + 30*2 + 40*3 + 50)
# Mismo bug que en SQL: la replicación distorsiona el SUM.

In [ ]:
# === ARREGLO: pre-agregar antes del SUM (DISTINCT por doc) ===
(
    df_join
    .select("doc_id", "articulos")                                   # nos quedamos con lo mínimo necesario
    .distinct()                                                       # "aplastamos" las réplicas
    .agg(F.countDistinct("doc_id").alias("docs"),
         F.sum("articulos").alias("total_articulos"))
    .show()
)
# Ahora sí: docs=5, total_articulos=150 ✅

**Importante:** en Spark este fan-out es **peor** que en SQL porque `join` implica shuffle. Si replicas filas innecesariamente, esas filas viajan por la red entre workers. **Regla de oro:** _agrega antes de hacer join cuando puedas_. Esto en Spark es una optimización de performance, no sólo de corrección.

## Bloque 7 — Window Functions

Las window functions de Spark son **conceptualmente idénticas** a las de SQL. La sintaxis cambia un poco:

```sql
ROW_NUMBER() OVER (PARTITION BY tipo ORDER BY articulos DESC) AS rk
```

↓

```python
from pyspark.sql.window import Window
w = Window.partitionBy("tipo").orderBy(F.desc("articulos"))
df.withColumn("rk", F.row_number().over(w))
```

In [ ]:
from pyspark.sql.window import Window                                # ventanas viven en su propio módulo

# Definimos una ventana: particionada por tipo, ordenada por número de artículos descendente
w = Window.partitionBy("tipo").orderBy(F.desc("articulos"))

(
    df_csv
    .withColumn("rk",   F.row_number().over(w))                       # ranking dentro de cada tipo
    .withColumn("dense", F.dense_rank().over(w))                      # dense_rank (sin huecos)
    .withColumn("max_tipo", F.max("articulos").over(
        Window.partitionBy("tipo")                                    # MAX OVER (PARTITION BY tipo)
    ))
    .orderBy("tipo", "rk")
    .show()
)

## Bloque 8 — Particiones y `explain()`

Una **partición** es un trozo del DataFrame que vive en un worker. El número de particiones determina el paralelismo y el coste de los shuffles.

- Pocas particiones → poco paralelismo, posible OOM en cada worker.
- Muchas particiones → overhead de scheduling, ficheros pequeños.
- Regla práctica: **2-4× el número de cores**, particiones de **~100-200 MB**.

Operaciones útiles:
- `df.rdd.getNumPartitions()` → cuántas tengo
- `df.repartition(N)` → fuerza N particiones (provoca shuffle)
- `df.coalesce(N)` → reduce particiones **sin shuffle** (solo agrupa locales)

In [ ]:
print(f"Particiones iniciales de df_csv: {df_csv.rdd.getNumPartitions()}")

df_8 = df_csv.repartition(8)                                          # forzamos 8 particiones (shuffle)
print(f"Tras repartition(8): {df_8.rdd.getNumPartitions()}")

df_1 = df_8.coalesce(1)                                               # juntamos a 1 partición sin shuffle
print(f"Tras coalesce(1):    {df_1.rdd.getNumPartitions()}")

In [ ]:
# Comparamos los planes físicos de un join
df_join_explain = df_docs.join(df_bridge, on="doc_id")
df_join_explain.explain(mode="formatted")                             # muestra el plan optimizado por Catalyst

Mira el plan: Spark elige un `BroadcastHashJoin` porque `df_bridge` es muy pequeño → en lugar de hacer shuffle, **manda una copia completa** del bridge a todos los workers. Es la **optimización más importante** para joins de tabla pequeña con tabla grande: **broadcast join**. Lo veremos en detalle en Week 3.

---

## Ejercicios

Todos los ejercicios usan los DataFrames `df_csv`, `df_docs` y `df_bridge` definidos arriba.

Para resolverlos, **importa lo que ya está importado arriba** (`F`, `Window`). Los huecos están marcados con `# Tu código aquí`.

### Ejercicio 1 — Conteo y promedio por tipo (groupBy)

A partir de `df_csv`, calcula para **cada tipo de documento**:
- el número de documentos,
- el promedio de artículos (redondeado a 1 decimal con `F.round(..., 1)`).

Ordena de mayor a menor número de documentos.

**Pista:** `groupBy(...).agg(F.count(...), F.round(F.avg(...), 1))`

In [ ]:
# EJERCICIO 1:
# Tu código aquí
(
    df_csv
    .groupBy("tipo")                          # decisión 1
    .agg(
        F.count("*").alias("num_docs"),                       # decisión 2
        F.round(F.avg("articulos"), 1).alias("media_articulos"),  # decisión 3
    )
    .orderBy(F.desc("num_docs"))                # decisión 4
    .show()
)


### Ejercicio 2 — Filtrar + transformar (filter + withColumn)

Del `df_csv`, devuelve los documentos publicados en **2024** (no 2025), añade una columna `año` con `F.year(F.col('fecha'))` y otra `tamaño` que sea `'grande'` si `articulos > 30`, `'medio'` si `articulos > 15`, `'pequeño'` en caso contrario.

**Pista:** `F.when(...).when(...).otherwise(...)` permite encadenar condiciones (como CASE WHEN en SQL).

In [ ]:
# EJERCICIO 2:
# Tu código aquí
(
    df_csv
    .filter(F.year(F.col("fecha")) == 2024)                                                # decisión 1
    .withColumn("año", F.year(F.col("fecha")))
    .withColumn(
        "tamaño",
        F.when(F.col("articulos") > 30, "grande")                 # decisión 3 — el más restrictivo primero
         .when(F.col("articulos") > 15, "medio")
         .otherwise("pequeño")
    )
    .show()
)


### Ejercicio 3 — Window Function (LAG)

A partir de `df_csv`, ordenando los documentos por fecha de publicación, calcula para cada uno:
- la fecha del documento **anterior** (`F.lag("fecha").over(w)`),
- los **días transcurridos** entre el documento anterior y el actual.

**Pistas:**
- `Window.orderBy("fecha")` (sin `partitionBy` → ventana sobre todo el DataFrame).
- Restar dos `DateType` en Spark devuelve días enteros si usas `F.datediff(fecha_actual, fecha_anterior)`.

In [ ]:
# EJERCICIO 3:
# Tu código aquí
from pyspark.sql.window import Window   # ya importado, recordatorio

w = Window.orderBy("fecha")                  # decisión 1

(
    df_csv
    .withColumn("fecha_anterior", F.lag("fecha").over(w))             # decisión 2
    .withColumn("dias_desde_anterior", F.datediff(F.col("fecha"), F.col("fecha_anterior")))      # decisión 3 — ¡orden!
    .orderBy("fecha")                                              # para verlo legible
    .show()
)


### Ejercicio 4 — Join + arreglar el fan-out

Usando `df_docs` y `df_bridge`, calcula el **número de temas** que tiene cada documento y el **número de artículos por tema** (un promedio justo: artículos del documento / número de temas).

Resultado esperado (orden de filas no importa):

| doc_id | titulo | articulos | num_temas | articulos_por_tema |
|---|---|---|---|---|
| 1 | Doc A | 10 | 1 | 10.00 |
| 2 | Doc B | 20 | 2 | 10.00 |
| 3 | Doc C | 30 | 2 | 15.00 |
| 4 | Doc D | 40 | 3 | 13.33 |
| 5 | Doc E | 50 | 1 | 50.00 |

**Pistas:**
- Primero agrega el `df_bridge` por `doc_id` para contar temas: `df_bridge.groupBy('doc_id').agg(F.count('*').alias('num_temas'))`.
- Luego haz `join` con `df_docs` y calcula la columna `articulos_por_tema`.
- **Conexión Week 1 → Week 2:** estás pre-agregando antes del join — exactamente el patrón limpio del fan-out.

In [ ]:
# EJERCICIO 4:
# Tu código aquí
# Paso 1: pre-agregar el bridge → 1 fila por doc_id
df_temas = (
    df_bridge
    .groupBy("doc_id")                                                # decisión 1
    .agg(F.count("*").alias("num_temas"))
)

(
    df_docs
    .join(df_temas, on="doc_id", how="inner") # Paso 2: join con docs (ahora SIN fan-out porque df_temas ya está colapsado)
    .withColumn(
        "articulos_por_tema",
        F.round(F.col("articulos") / F.col("num_temas"), 2)                      # decisión 3
    )
    .select("doc_id", "titulo", "articulos", "num_temas", "articulos_por_tema")  # decisión 4
    .orderBy("doc_id")
    .show()
)


---

## Cierre

Cuando hayas terminado los ejercicios, ejecuta esta celda para cerrar la `SparkSession` y liberar recursos. (No es estrictamente necesario, pero es buena costumbre.)

In [ ]:
spark.stop()